# Lezione 02 — Riproducibilità

**Corso:** MLOps e Monitoraggio con Python · **Blocco 1:** Il problema e le basi
**Tempo stimato:** 100 minuti · **Capitolo del manuale:** `manual/02_reproducibilidad_IT.md`

## Cosa vedrai in questa lezione

- Perché **"sulla mia macchina funziona"** è un sintomo di un problema di riproducibilità, non una scusa.
- **Dipendenze bloccate**: generare un `requirements.txt` e un `pyproject.toml` reali, con le versioni esatte installate in questo ambiente.
- **Tutte** le fonti di casualità di un piccolo progetto (NumPy, divisione train/test, il modello stesso) e quali si dimenticano più spesso.
- Una **struttura di cartelle** con responsabilità chiare (`src/`, `config/`, `tests/`), ispirata a Cookiecutter Data Science.
- **Separare la configurazione dal codice** con un `config.yaml`.
- La rifattorizzazione guidata, con codice eseguito davvero, da `datos/notebook_sucio.ipynb` a `proyecto_churn/`: la dimostrazione finale confronta due esecuzioni della stessa *pipeline* e prova che una è riproducibile e l'altra no.

Questo notebook crea una cartella di lavoro effimera, `proyecto_churn/`, accanto a questo stesso notebook (si ricrea vuota ogni volta che esegui la Cella 0, così parti sempre da zero).

## Prerequisiti

- Librerie: `numpy`, `pandas`, `scikit-learn` e `PyYAML`. La cella seguente installa solo ciò che manca (funziona allo stesso modo in Google Colab).
- La cartella `datos/` del corso, con `notebook_sucio.ipynb` all'interno (la apriamo solo come riferimento, come nella lezione 1).

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta.

---
## Installazione

In [ ]:
import subprocess, sys


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("PyYAML", "yaml")
print("Librerie pronte.")

---
## Cella 0 — Setup

Carichiamo `notebook_sucio.ipynb` come riferimento (senza eseguirlo, come nella lezione 1) e prepariamo `proyecto_churn/`, la cartella di lavoro effimera che riempiremo durante la lezione.

In [ ]:
import os
import re
import sys
import json
import shutil
import subprocess
import importlib.util
from importlib.metadata import version

import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

RUTA_SUCIO = cargar_datos("notebook_sucio.ipynb")
sucio = json.load(open(RUTA_SUCIO, encoding="utf-8"))
print("Notebook sporco caricato (solo come riferimento, non si esegue):", RUTA_SUCIO)

# Cartella di lavoro effimera: si ricrea vuota a ogni esecuzione
WORK = "proyecto_churn"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK)
print("Cartella di lavoro pronta:", os.path.abspath(WORK))

---
## 1. "Sulla mia macchina funziona" non è una scusa

Riproduciamo la cella 4 del notebook sporco (senza la parte del file "extra") in una funzione, e la chiamiamo due volte di seguito, nello stesso notebook, senza fissare nessun seme.

In [ ]:
def generar_datos_sin_semilla(n=400):
    """Stessa logica della cella 4 di notebook_sucio.ipynb, senza nessun seme."""
    datos = pd.DataFrame({
        "antiguedad_meses": np.random.randint(1, 72, n),
        "uso_mensual_horas": np.random.gamma(2, 5, n),
        "tickets_soporte": np.random.poisson(1.5, n),
        "factura_mensual": np.random.normal(35, 12, n),
    })
    prob_base = 0.05 + 0.5 * (datos["tickets_soporte"] > 2) + 0.5 * (datos["uso_mensual_horas"] > 18)
    datos["abandono"] = (np.random.rand(n) < prob_base.clip(0.02, 0.95)).astype(int)
    return datos


datos_1 = generar_datos_sin_semilla(400)
datos_2 = generar_datos_sin_semilla(400)

print("righe generate in ogni chiamata:", len(datos_1), len(datos_2))
print("le due chiamate danno lo stesso DataFrame?", datos_1.equals(datos_2))
assert not datos_1.equals(datos_2)

**Cosa osservi?** Le due chiamate usano esattamente la stessa funzione e lo stesso argomento (`n=400`), eppure il `DataFrame` non è identico (`datos_1.equals(datos_2)` dà `False`). Nessuna macchina ha colpa: il problema è che `generar_datos_sin_semilla` non fissa mai da quale stato parte `np.random`. È esattamente lo stesso difetto della cella 4 del notebook sporco.

---
## 2. Dipendenze bloccate: `requirements.txt` e `pyproject.toml`

Generiamo entrambi i file con le versioni **realmente installate** in questo ambiente, usando `importlib.metadata.version`, senza scriverle a mano.

In [ ]:
PAQUETES = [("numpy", "numpy"), ("pandas", "pandas"), ("scikit-learn", "scikit-learn"), ("PyYAML", "PyYAML")]
versiones = {nombre_pip: version(nombre_pip) for nombre_pip, _ in PAQUETES}

lineas_requirements = [f"{paquete}=={ver}" for paquete, ver in versiones.items()]
with open(os.path.join(WORK, "requirements.txt"), "w") as f:
    f.write("\n".join(lineas_requirements) + "\n")

print("requirements.txt generato:")
print(open(os.path.join(WORK, "requirements.txt")).read())

In [ ]:
dependencias_toml = ",\n    ".join(f'"{paquete}=={ver}"' for paquete, ver in versiones.items())
pyproject = f"""[project]
name = "proyecto_churn"
version = "0.1.0"
dependencies = [
    {dependencias_toml}
]
"""
with open(os.path.join(WORK, "pyproject.toml"), "w") as f:
    f.write(pyproject)

print("pyproject.toml generato (formato alternativo, stessa informazione):")
print(open(os.path.join(WORK, "pyproject.toml")).read())

**Cosa osservi?** Entrambi i file fissano le stesse 4 versioni esatte, in due formati diversi. Nessuno dei due dice nulla sulla versione di Python, sul sistema operativo o sulle librerie di sistema: è quello che risolve Docker, nella lezione 8 (qui lo menzioniamo soltanto).

---
## 3. Fissare i semi in tutte le fonti di casualità

Riscriviamo `generar_datos_sin_semilla` con un generatore proprio (`np.random.default_rng(seed)`) che riceve il seme come parametro, invece di toccare lo stato globale.

In [ ]:
def generar_datos(n, seed):
    """Stessa logica, con generatore proprio e seme esplicito."""
    rng = np.random.default_rng(seed)
    datos = pd.DataFrame({
        "antiguedad_meses": rng.integers(1, 72, n),
        "uso_mensual_horas": rng.gamma(2, 5, n),
        "tickets_soporte": rng.poisson(1.5, n),
        "factura_mensual": rng.normal(35, 12, n),
    })
    prob_base = 0.05 + 0.5 * (datos["tickets_soporte"] > 2) + 0.5 * (datos["uso_mensual_horas"] > 18)
    datos["abandono"] = (rng.random(n) < prob_base.clip(0.02, 0.95)).astype(int)
    return datos


misma_semilla_1 = generar_datos(400, seed=42)
misma_semilla_2 = generar_datos(400, seed=42)
otra_semilla = generar_datos(400, seed=7)

print("stesso seme → stesso DataFrame:", misma_semilla_1.equals(misma_semilla_2))
print("seme diverso → DataFrame diverso:", not misma_semilla_1.equals(otra_semilla))
assert misma_semilla_1.equals(misma_semilla_2)
assert not misma_semilla_1.equals(otra_semilla)

**Cosa osservi?** Con `generar_datos(400, seed=42)`, due chiamate consecutive danno lo stesso `DataFrame` byte per byte; con `seed=7`, il risultato cambia, come è prevedibile. Fissare il seme non elimina la casualità: la trasforma in **pseudocasualità controllata** (sezione 3 del capitolo). Manca ancora fissare la divisione train/test e il modello — lo facciamo nella sezione 6.

---
## 4. Struttura di cartelle: `src/`, `config/`, `tests/`

Creiamo la struttura minima dentro `proyecto_churn/`, ispirata a Cookiecutter Data Science (capitolo, sezione 4), senza copiarne il modello completo.

In [ ]:
for carpeta in ("src", "config", "tests"):
    os.makedirs(os.path.join(WORK, carpeta), exist_ok=True)

print(f"Struttura di {WORK}/:")
for raiz, subcarpetas, archivos in os.walk(WORK):
    subcarpetas.sort()
    nivel = raiz.replace(WORK, "").count(os.sep)
    print("  " * nivel + os.path.basename(raiz or WORK) + "/")
    for archivo in sorted(archivos):
        print("  " * (nivel + 1) + archivo)

**Cosa osservi?** `proyecto_churn/` ha già tre cartelle con responsabilità chiara e nessun file sparso nella radice a parte `requirements.txt` e `pyproject.toml`. Confrontalo con il "Progetto A" del capitolo 1 (sezione 3.1): un unico `.ipynb` dove tutto — dati, addestramento, valutazione — vive nello stesso file.

---
## 5. Separare la configurazione dal codice

Nessun iperparametro né seme resta scritto dentro una funzione: vivono tutti in `config/config.yaml`.

In [ ]:
config = {
    "n_filas": 400,
    "test_size": 0.25,
    "hiperparametros": {"n_estimators": 100, "max_depth": 5},
    "semillas": {"datos": 42, "split": 42, "modelo": 42},
}

with open(os.path.join(WORK, "config", "config.yaml"), "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("config.yaml scritto. Contenuto:")
print(open(os.path.join(WORK, "config", "config.yaml")).read())

config_releido = yaml.safe_load(open(os.path.join(WORK, "config", "config.yaml")))
print("ciò che è stato letto è uguale a ciò che è stato scritto?", config_releido == config)
assert config_releido == config

**Cosa osservi?** Il dizionario `config` che è stato scritto e quello riletto da disco sono esattamente uguali: `config.yaml` è una rappresentazione fedele, in testo semplice, dei parametri del progetto. Cambiare `"n_estimators": 100` in `200` significherebbe modificare una riga di questo file, senza toccare `src/` (che non esiste ancora: la creiamo nella sezione seguente).

---
## 6. La logica in `src/pipeline.py`, e la prova finale di riproducibilità

Scriviamo il codice della *pipeline* in un file reale, `src/pipeline.py` (non in una cella isolata), con tre funzioni: `generar_datos`, `dividir` e `entrenar`, tutte con parametri espliciti e nessuna variabile globale. Poi lo importiamo e lo eseguiamo due volte.

In [ ]:
codigo_pipeline = '''
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


def generar_datos(n, seed):
    rng = np.random.default_rng(seed)
    datos = pd.DataFrame({
        "antiguedad_meses": rng.integers(1, 72, n),
        "uso_mensual_horas": rng.gamma(2, 5, n),
        "tickets_soporte": rng.poisson(1.5, n),
        "factura_mensual": rng.normal(35, 12, n),
    })
    prob_base = 0.05 + 0.5 * (datos["tickets_soporte"] > 2) + 0.5 * (datos["uso_mensual_horas"] > 18)
    datos["abandono"] = (rng.random(n) < prob_base.clip(0.02, 0.95)).astype(int)
    return datos


def dividir(datos, config):
    columnas = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
    X, y = datos[columnas], datos["abandono"]
    return train_test_split(X, y, test_size=config["test_size"], random_state=config["semillas"]["split"])


def entrenar(X_train, y_train, config):
    modelo = RandomForestClassifier(
        n_estimators=config["hiperparametros"]["n_estimators"],
        max_depth=config["hiperparametros"]["max_depth"],
        random_state=config["semillas"]["modelo"],
    )
    modelo.fit(X_train, y_train)
    return modelo


def pipeline_completo(config):
    datos = generar_datos(config["n_filas"], config["semillas"]["datos"])
    X_train, X_test, y_train, y_test = dividir(datos, config)
    modelo = entrenar(X_train, y_train, config)
    exactitud = modelo.score(X_test, y_test)
    predicciones = modelo.predict(X_test)
    return modelo, exactitud, predicciones
'''

with open(os.path.join(WORK, "src", "pipeline.py"), "w") as f:
    f.write(codigo_pipeline)

spec = importlib.util.spec_from_file_location("pipeline", os.path.join(WORK, "src", "pipeline.py"))
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)
print("src/pipeline.py scritto e importato.")

In [ ]:
# Esecuzione 1 ed esecuzione 2 della stessa pipeline, con la stessa config: devono coincidere
_, exactitud_1, predicciones_1 = pipeline.pipeline_completo(config)
_, exactitud_2, predicciones_2 = pipeline.pipeline_completo(config)

print("pipeline_completo — accuratezza esecuzione 1:", exactitud_1)
print("pipeline_completo — accuratezza esecuzione 2:", exactitud_2)
print("stessa accuratezza nelle due esecuzioni?", exactitud_1 == exactitud_2)
print("stesse predizioni nelle due esecuzioni?", list(predicciones_1) == list(predicciones_2))
assert exactitud_1 == exactitud_2
assert list(predicciones_1) == list(predicciones_2)

**Cosa osservi?** Con i tre semi di `config.yaml` fissati (dati, divisione e modello), `pipeline_completo` dà esattamente la stessa accuratezza e le stesse predizioni nelle due esecuzioni. Ora confrontiamo con una versione che dimentica solo di fissare il seme del modello — una sola chiamata, tra varie, senza `random_state` — per vedere se resta riproducibile.

In [ ]:
def entrenar_sin_semilla(X_train, y_train, config):
    """Uguale a entrenar(), ma "dimentica" random_state nel modello."""
    modelo = RandomForestClassifier(
        n_estimators=config["hiperparametros"]["n_estimators"],
        max_depth=config["hiperparametros"]["max_depth"],
    )
    modelo.fit(X_train, y_train)
    return modelo


# Dati e divisione HANNO seme (2 delle 3 fonti della sezione 3 sono controllate)
datos_config = pipeline.generar_datos(config["n_filas"], config["semillas"]["datos"])
X_train, X_test, y_train, y_test = pipeline.dividir(datos_config, config)

# Simuliamo "macchina A" e "macchina B": ciascuna eredita uno stato globale di NumPy diverso,
# qualcosa di reale e fuori dal tuo controllo, che qui fissiamo solo perché questa dimostrazione sia riproducibile nel manuale
np.random.seed(1001)
modelo_a = entrenar_sin_semilla(X_train, y_train, config)
exactitud_a = modelo_a.score(X_test, y_test)

np.random.seed(2002)
modelo_b = entrenar_sin_semilla(X_train, y_train, config)
exactitud_b = modelo_b.score(X_test, y_test)

print("senza seme nel modello — accuratezza «macchina A»:", exactitud_a)
print("senza seme nel modello — accuratezza «macchina B»:", exactitud_b)
print("stessa accuratezza nelle due «macchine»?", exactitud_a == exactitud_b)
assert exactitud_a != exactitud_b

**Cosa osservi?** Con gli stessi dati e la stessa divisione (due delle tre fonti di casualità controllate), dimenticare `random_state` in una sola chiamata — quella del modello — basta perché "macchina A" e "macchina B" addestrino modelli diversi e ottengano accuratezze diverse. È la sezione 3.1 del capitolo fatta codice: **un `random_state` che manca in una sola chiamata, in un progetto con più passaggi**, è il caso più insidioso perché il resto della *pipeline* sembra controllato.

---
### 6.1 Una prova automatica, con `pytest`

Per finire, scriviamo `tests/test_pipeline.py`: una prova che fallisce se `pipeline_completo` smette di essere riproducibile. La eseguiamo con `pytest` per davvero, in un sottoprocesso, esattamente come verrebbe eseguita nella lezione 6 o in un flusso di integrazione continua (lezione 9).

In [ ]:
codigo_test = '''
import os
import sys
import yaml

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from src.pipeline import pipeline_completo


def test_pipeline_es_reproducible():
    ruta_config = os.path.join(os.path.dirname(__file__), "..", "config", "config.yaml")
    with open(ruta_config) as f:
        config = yaml.safe_load(f)
    _, exactitud_1, predicciones_1 = pipeline_completo(config)
    _, exactitud_2, predicciones_2 = pipeline_completo(config)
    assert exactitud_1 == exactitud_2
    assert list(predicciones_1) == list(predicciones_2)
'''

with open(os.path.join(WORK, "tests", "test_pipeline.py"), "w") as f:
    f.write(codigo_test)

resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=WORK, capture_output=True, text=True)
coincidencias = re.findall(r"(\d+) passed", resultado.stdout)
n_pasadas = coincidencias[0] if coincidencias else "0"
print("tests/test_pipeline.py eseguito con pytest. Prove superate:", n_pasadas)
assert resultado.returncode == 0
assert n_pasadas == "1"

**Cosa osservi?** `pytest` conferma, in modo automatico, ciò che abbiamo già visto a mano: la prova passa (`1 passed`) perché `pipeline_completo`, con i tre semi fissati in `config.yaml`, è riproducibile. Da ora in poi, se qualcuno introduce per sbaglio un `RandomForestClassifier()` senza `random_state` dentro `src/pipeline.py`, questa stessa prova fallirebbe e lo segnalerebbe, senza dipendere dal fatto che qualcuno se ne accorga a occhio.

---
## Riepilogo della lezione

- **"Sulla mia macchina funziona"** è un sintomo: se lo stesso codice con gli stessi dati dà risultati diversi, il problema è nel codice, non in chi lo esegue.
- Le **dipendenze bloccate** (`requirements.txt` o `pyproject.toml`, con versioni esatte) sono necessarie ma non sufficienti: fissano i pacchetti, non l'ambiente completo (quello è Docker, lezione 8).
- Un piccolo progetto ha **almeno tre fonti di casualità** (NumPy, divisione train/test, il modello); basta dimenticarne una — spesso lo *shuffle* interno o l'inizializzazione dei pesi del modello — per perdere completamente la riproducibilità.
- Una **struttura di cartelle** con responsabilità chiare (`src/`, `config/`, `tests/`), ispirata a Cookiecutter Data Science, e una **configurazione separata dal codice** (`config.yaml`) fanno sì che cambiare un parametro non obblighi a toccare il codice.
- `proyecto_churn/` dimostra, con `pytest` di mezzo, che la stessa *pipeline* con i tre semi fissati dà lo stesso risultato in due esecuzioni, mentre dimenticarne solo uno (quello del modello) la rompe già.

*Continua con la lezione 3: Versionamento di dati e modelli.*